# Generate the simulated network dataset

Creates `data/simulated_network_data.csv` (330 rows) with a fixed random seed so the dataset is reproducible.


| Column | Distribution |
|---|---|
| `timestamp` | Starts 2026-01-01; gaps between observations are uniform random 5-120 minutes |
| `node_id` | Uniform choice from 101-105 |
| `latency_ms` | Normal(mean 50, sd 15), clipped to 5-150 |
| `response_time_ms` | `latency_ms` + Gamma(shape 2, scale 60) processing delay |
| `throughput_MBps` | Normal(mean 5, sd 1.5), floored at 0.1 |
| `status` | Active 85% / Inactive 10% / Error 5% |

Because nodes, status and measurements are independent, any differences between nodes in the analysis are random noise by construction.

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Make the simulation reproducible
np.random.seed(42)

In [3]:
n = 330

# Generate timestamps at random intervals
start= pd.Timestamp("2026-01-01 00:00:00")
intervals = np.random.randint(5, 121, size=n) #minutes
timestamps = start + pd.to_timedelta(
    np.cumsum(intervals), unit="min"
)

In [4]:
# Generate network measurements
node_ids = np.random.choice(
    [101, 102, 103, 104, 105], size=n
)

latency = np.clip(
    np.random.normal(loc=50, scale=15, size=n),
    5, 150
)

In [5]:
# Response time includes latency and additional processing delay
response_time = latency + np.random.gamma(
    shape=2, scale=60, size=n
)

throughput = np.clip(
    np.random.normal(loc=5, scale=1.5, size=n),
    0.1, None
)

In [6]:
# Assign statuses probabilistically
statuses = np.random.choice(
    ["Active", "Inactive", "Error"],
    size=n,
    p=[0.85, 0.10, 0.05]
)

In [7]:
# Build the dataset
df = pd.DataFrame({
    "timestamp": timestamps,
    "node_id": node_ids,
    "latency_ms": latency.round(2),
    "response_time_ms": response_time.round(2),
    "throughput_MBps": throughput.round(2),
    "status": statuses
})

In [8]:
# Save the simulated dataset
df.to_csv("../data/simulated_network_data.csv", index=False)

print(df.head())
print(df.dtypes)

            timestamp  node_id  latency_ms  response_time_ms  throughput_MBps  \
0 2026-01-01 01:47:00      102       65.91            135.07             2.55   
1 2026-01-01 02:43:00      103       53.56             62.77             7.37   
2 2026-01-01 04:20:00      104       45.24            170.60             2.84   
3 2026-01-01 04:39:00      102       56.88            277.18             3.17   
4 2026-01-01 06:30:00      101       52.79            483.79             3.47   

     status  
0    Active  
1    Active  
2    Active  
3    Active  
4  Inactive  
timestamp           datetime64[us]
node_id                      int64
latency_ms                 float64
response_time_ms           float64
throughput_MBps            float64
status                         str
dtype: object
